In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from app.ingestion.loader import add_ids, clean_house_size, filter_columns, drop_incomplete_rows, process_city, load_all_properties, save_processed

In [2]:
df1 = pd.read_csv("../data/raw/Indian_housing_Mumbai_data.csv")
df2 = pd.read_csv("../data/raw/Indian_housing_Pune_data.csv")
df3 = pd.read_csv("../data/raw/Indian_housing_Delhi_data.csv")

df1.head()

,house_type,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,isNegotiable,priceSqFt,verificationDate,description,SecurityDeposit,Status
0,2 BHK Apartment,"1,180 sq ft",Ulwe,Mumbai,18.971046,73.017723,19000,INR,2.0,NaN,NaN,NaN,Posted 2 years ago,It has area of 1180 sqft and is available at a...,No Deposit,Furnished
1,2 BHK Apartment,"1,120 sq ft",Ulwe,Mumbai,18.966377,73.012802,12500,INR,2.0,NaN,NaN,NaN,Posted 2 years ago,Well designed 2 bhk multistorey apartment is a...,No Deposit,Unfurnished
2,2 BHK Apartment,"1,200 sq ft",Panvel,Mumbai,18.965979,73.124649,22000,INR,2.0,NaN,NaN,NaN,Posted a month ago,A spacious 2 bhk multistorey apartment is avai...,No Deposit,Unfurnished
3,2 BHK Apartment,750 sq ft,Kandivali West,Mumbai,19.213598,72.833633,28500,INR,2.0,NaN,NaN,NaN,Posted 3 years ago,It has a built-up area of 750 sqft and is avai...,No Deposit,Unfurnished
4,3 BHK Apartment,"1,400 sq ft",Chembur,Mumbai,19.062933,72.892395,50000,INR,3.0,NaN,NaN,NaN,Posted 3 years ago,The house is semi-furnished. It has power back...,No Deposit,Semi-Furnished


In [3]:
df1['city'].value_counts()

df1[df1["city"] == "Hisar"][[
    "city",
    "location",
    "house_type",
    "price",
    "description",
    # "id"
]]

,city,location,house_type,price,description
2724,Hisar,DN Nagar,2 BHK Apartment,70000,It has an area of 850 sqft with a carpet area ...
2842,Hisar,Borivali (West),1 BHK Apartment,22000,This spacious 1 bhk multistorey apartment is a...
3548,Hisar,Lokhandwala,3 BHK Apartment,130000,Available 3 bhk flat at Andheri West\nFamily a...
3559,Hisar,Lokhandwala,3 BHK Apartment,130000,Available 3 bhk flat at Andheri West\nFamily a...
3849,Hisar,DN Nagar,2 BHK Apartment,55000,It has area of 1055 sqft and is available at a...
3945,Hisar,DN Nagar,1 BHK Apartment,35000,It has an area of 650 sqft with a carpet area ...
4409,Hisar,Versova,1 BHK Apartment,7000,A 1 bhk property is available for rental in Zo...
4420,Hisar,Versova,1 BHK Apartment,7000,A 1 bhk property is available for rental in Zo...


In [4]:
print(df1.info())
print("=" * 20)
df1["house_type"].value_counts().head()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   house_type        5000 non-null   str    
 1   house_size        5000 non-null   str    
 2   location          5000 non-null   str    
 3   city              5000 non-null   str    
 4   latitude          5000 non-null   float64
 5   longitude         5000 non-null   float64
 6   price             5000 non-null   int64  
 7   currency          5000 non-null   str    
 8   numBathrooms      4986 non-null   float64
 9   numBalconies      733 non-null    float64
 10  isNegotiable      571 non-null    str    
 11  priceSqFt         0 non-null      float64
 12  verificationDate  5000 non-null   str    
 13  description       4922 non-null   str    
 14  SecurityDeposit   5000 non-null   str    
 15  Status            5000 non-null   str    
dtypes: float64(5), int64(1), str(10)
memory usage: 625.1 

house_type
2 BHK Apartment           2150
1 BHK Apartment           1580
3 BHK Apartment            896
1 RK Studio Apartment      168
4 BHK Apartment            133
Name: count, dtype: int64

In [5]:
print(df2.info())
print("=" * 20)
df2["house_type"].value_counts().head()

<class 'pandas.DataFrame'>
RangeIndex: 3910 entries, 0 to 3909
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   house_type        3910 non-null   str    
 1   house_size        3910 non-null   str    
 2   location          3910 non-null   str    
 3   city              3910 non-null   str    
 4   latitude          3910 non-null   float64
 5   longitude         3910 non-null   float64
 6   price             3910 non-null   int64  
 7   currency          3910 non-null   str    
 8   numBathrooms      3893 non-null   float64
 9   numBalconies      1821 non-null   float64
 10  isNegotiable      526 non-null    str    
 11  priceSqFt         0 non-null      float64
 12  verificationDate  3910 non-null   str    
 13  description       3442 non-null   str    
 14  SecurityDeposit   3910 non-null   str    
 15  Status            3910 non-null   str    
dtypes: float64(5), int64(1), str(10)
memory usage: 488.9 

house_type
2 BHK Apartment             2009
1 BHK Apartment              853
3 BHK Apartment              619
1 RK Studio Apartment        128
1 BHK Independent House       60
Name: count, dtype: int64

In [6]:
print(df3.info())
print("=" * 20)
df3["house_type"].value_counts().head()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   house_type        5000 non-null   str    
 1   house_size        5000 non-null   str    
 2   location          5000 non-null   str    
 3   city              5000 non-null   str    
 4   latitude          5000 non-null   float64
 5   longitude         5000 non-null   float64
 6   price             5000 non-null   int64  
 7   currency          5000 non-null   str    
 8   numBathrooms      4975 non-null   float64
 9   numBalconies      2737 non-null   float64
 10  isNegotiable      179 non-null    str    
 11  priceSqFt         0 non-null      float64
 12  verificationDate  5000 non-null   str    
 13  description       4715 non-null   str    
 14  SecurityDeposit   5000 non-null   str    
 15  Status            5000 non-null   str    
dtypes: float64(5), int64(1), str(10)
memory usage: 625.1 

house_type
3 BHK Independent Floor     1567
4 BHK Independent Floor      783
2 BHK Independent Floor      509
3 BHK Apartment              322
4 BHK Villa                  258
Name: count, dtype: int64

In [7]:
clean_house_size("1,180 sq ft")
clean_house_size("450 sq ft")
clean_house_size("1,90,000 sq ft")

190000.0

In [8]:
clean_house_size("450 sq ft")
clean_house_size("1,90,000 sq ft")

190000.0

In [9]:
test_df = add_ids(df1.copy(), "Pune")
test_df[["id"]].head()

,id
0,Pune_0
1,Pune_1
2,Pune_2
3,Pune_3
4,Pune_4


In [10]:
filtered = filter_columns(df1.copy())
filtered.head()

,house_type,house_size,location,city,price,numBathrooms,description,SecurityDeposit,Status
0,2 BHK Apartment,"1,180 sq ft",Ulwe,Mumbai,19000,2.0,It has area of 1180 sqft and is available at a...,No Deposit,Furnished
1,2 BHK Apartment,"1,120 sq ft",Ulwe,Mumbai,12500,2.0,Well designed 2 bhk multistorey apartment is a...,No Deposit,Unfurnished
2,2 BHK Apartment,"1,200 sq ft",Panvel,Mumbai,22000,2.0,A spacious 2 bhk multistorey apartment is avai...,No Deposit,Unfurnished
3,2 BHK Apartment,750 sq ft,Kandivali West,Mumbai,28500,2.0,It has a built-up area of 750 sqft and is avai...,No Deposit,Unfurnished
4,3 BHK Apartment,"1,400 sq ft",Chembur,Mumbai,50000,3.0,The house is semi-furnished. It has power back...,No Deposit,Semi-Furnished


In [11]:
dropped = drop_incomplete_rows(filtered)
dropped.info()

<class 'pandas.DataFrame'>
Index: 4908 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   house_type       4908 non-null   str    
 1   house_size       4908 non-null   str    
 2   location         4908 non-null   str    
 3   city             4908 non-null   str    
 4   price            4908 non-null   int64  
 5   numBathrooms     4908 non-null   float64
 6   description      4908 non-null   str    
 7   SecurityDeposit  4908 non-null   str    
 8   Status           4908 non-null   str    
dtypes: float64(1), int64(1), str(7)
memory usage: 383.4 KB


In [12]:
processed_df = process_city("../data/raw/Indian_housing_Delhi_data.csv", "Mumbai")
processed_df.head()

,house_type,house_size,location,city,price,numBathrooms,description,SecurityDeposit,Status,id
0,1 RK Studio Apartment,400.0,Kalkaji,Mumbai,22000,1.0,"Fully furnished, loaded with amenities & gadge...",No Deposit,Furnished,Mumbai_0
1,1 RK Studio Apartment,400.0,Mansarover Garden,Mumbai,20000,1.0,Here is an excellent 1 BHK Independent Floor a...,No Deposit,Furnished,Mumbai_1
2,2 BHK Independent Floor,500.0,Uttam Nagar,Mumbai,8500,1.0,"Zero Brokerage.\n\n2 Room set, Govt bijali Met...",No Deposit,Semi-Furnished,Mumbai_2
3,3 BHK Independent House,1020.0,Model Town,Mumbai,48000,3.0,Itâs a 3 bhk independent house situated in M...,No Deposit,Furnished,Mumbai_3
4,2 BHK Apartment,810.0,Sector 13 Rohini,Mumbai,20000,2.0,Well designed 2 bhk multistorey apartment is a...,No Deposit,Unfurnished,Mumbai_4


In [13]:
processed_df.info()

<class 'pandas.DataFrame'>
Index: 4691 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   house_type       4691 non-null   str    
 1   house_size       4691 non-null   float64
 2   location         4691 non-null   str    
 3   city             4691 non-null   str    
 4   price            4691 non-null   int64  
 5   numBathrooms     4691 non-null   float64
 6   description      4691 non-null   str    
 7   SecurityDeposit  4691 non-null   str    
 8   Status           4691 non-null   str    
 9   id               4691 non-null   str    
dtypes: float64(2), int64(1), str(7)
memory usage: 403.1 KB


In [14]:
processed_df.isna().sum()

house_type         0
house_size         0
location           0
city               0
price              0
numBathrooms       0
description        0
SecurityDeposit    0
Status             0
id                 0
dtype: int64

In [15]:
df = load_all_properties("../data/raw")

df.shape
df["city"].value_counts()

city
Mumbai    4908
Delhi     4691
Pune      3425
Name: count, dtype: int64

In [16]:
df["id"].head()

0    Delhi_0
1    Delhi_1
2    Delhi_2
3    Delhi_3
4    Delhi_4
Name: id, dtype: str

In [17]:
df["id"].tail()

3905    Pune_3905
3906    Pune_3906
3907    Pune_3907
3908    Pune_3908
3909    Pune_3909
Name: id, dtype: str

In [18]:
df = load_all_properties("../data/raw")

print("Shape:", df.shape)
print("Cities:")
print(df["city"].value_counts())

print("\nNulls:")
print(df.isna().sum())

print("\nDuplicate IDs:", df["id"].duplicated().sum())

print("\nHouse size type:", df["house_size"].dtype)

Shape: (13024, 10)
Cities:
city
Mumbai    4908
Delhi     4691
Pune      3425
Name: count, dtype: int64

Nulls:
house_type         0
house_size         0
location           0
city               0
price              0
numBathrooms       0
description        0
SecurityDeposit    0
Status             0
id                 0
dtype: int64

Duplicate IDs: 0

House size type: float64


In [19]:
save_processed(df, "../data/processed/properties.jsonl")

In [20]:
from pathlib import Path

path = Path("../data/processed/properties.jsonl")

print("Exists:", path.exists())
print("Lines:", sum(1 for _ in path.open()))

Exists: True
Lines: 13024


In [21]:
from app.embeddings.text_builder import build_property_text

sample = df.iloc[0]

text = build_property_text(sample)

print(text)

1 RK Studio Apartment  in Kalkaji, Delhi. 400.0 sq ft. Monthly rent: 22000. 1.0 bathrooms. Fully furnished, loaded with amenities & gadgets- 1RK + lobby Set, with all facilities, Park facing with front wide open terrace & backside Balcony, fully secured. Walkable distance to metro & market. Free daily room cleaning, 24hours free watersupply, Ideal accomodation for male professional working executive. More About This Property This is a modern and stylish home available for affordable rent in Delhi. It is a 1 BHK Independent Floor situated at Kalkaji. Designed to meet your lifestyle needs, the Independent Floor provides a range of modern amenities for the comfort of the residents. It is Fully Furnished. Modern home seekers will experience a comfortable living in this 1 BHK property. The Independent Floor is on 3 floor, and the total number of floors is 3. The 1 BHK unit is elegantly designed to meet your housing needs. The Independent Floor has 1 bedroom. It has 1 bathroom. The 1 BHK uni